# Flight Price Prediction - Sample Model Training

This notebook demonstrates training and evaluating multiple machine learning models on flight price prediction data.

**Objective**: Train various regression models and compare their performance on predicting flight ticket prices.

## 1. Import Libraries and Configuration

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')

print("="*80)
print("FLIGHT PRICE PREDICTION - SAMPLE MODEL TRAINING")
print("="*80)

FLIGHT PRICE PREDICTION - SAMPLE MODEL TRAINING


## 2. Load Training Data

In [2]:
print("\n[1] Loading training data...")
try:
    df_train = pd.read_csv('Flight_Data_Train_Clean.csv')
    df_test = pd.read_csv('Flight_Data_Test_Clean.csv')
    print(f"✓ Training set shape: {df_train.shape}")
    print(f"✓ Test set shape: {df_test.shape}")
except FileNotFoundError:
    print("✗ Error: CSV files not found. Make sure you're in the project directory.")
    exit(1)

# Display sample features
print(f"\nDataset Features: {df_train.columns.tolist()}")
print(f"\nFirst 5 rows of training data:")
df_train.head()


[1] Loading training data...
✓ Training set shape: (240122, 13)
✓ Test set shape: (60031, 13)

Dataset Features: ['airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'duration', 'days_left', 'route', 'duration_category', 'price']

First 5 rows of training data:


,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,route,duration_category,price
0,Air_India,AI-424,Kolkata,Afternoon,one,Morning,Chennai,Economy,19.42,6,Kolkata → Chennai,Very Long (>12h),13524
1,Indigo,6E-2193,Delhi,Afternoon,two_or_more,Night,Chennai,Economy,7.00,13,Delhi → Chennai,Long (6-12h),9940
2,Air_India,AI-768,Kolkata,Afternoon,one,Afternoon,Chennai,Business,21.17,44,Kolkata → Chennai,Very Long (>12h),55983
3,Vistara,UK-876,Hyderabad,Night,one,Early_Morning,Bangalore,Economy,10.25,11,Hyderabad → Bangalore,Long (6-12h),7927
4,Vistara,UK-774,Kolkata,Night,one,Night,Chennai,Business,26.50,5,Kolkata → Chennai,Very Long (>12h),55502


## 3. Data Preprocessing

In [3]:
print("\n[2] Preprocessing data...")

# Separate features and target
X_train = df_train.drop('price', axis=1)
y_train = df_train['price']
X_test = df_test.drop('price', axis=1)
y_test = df_test['price']

print(f"Features shape: {X_train.shape}")
print(f"Target variable: price (min: ₹{y_train.min():.2f}, max: ₹{y_train.max():.2f})")

# Identify categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nCategorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")


[2] Preprocessing data...
Features shape: (240122, 12)
Target variable: price (min: ₹1105.00, max: ₹123071.00)

Categorical features (10): ['airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'route', 'duration_category']
Numerical features (2): ['duration', 'days_left']


## 4. Encode Categorical Features

In [4]:
print("\n[3] Encoding categorical features...")
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Get all unique values from both train and test
    all_values = pd.concat([X_train[col], X_test[col]]).astype(str).unique()
    le.fit(all_values)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le
print(f"✓ Encoded {len(categorical_cols)} categorical features")


[3] Encoding categorical features...
✓ Encoded 10 categorical features


## 5. Scale Numerical Features

In [5]:
print("\n[4] Scaling numerical features...")
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])
print(f"✓ Scaled {len(numerical_cols)} numerical features")


[4] Scaling numerical features...
✓ Scaled 2 numerical features


## 6. Train Machine Learning Models

In [6]:
print("\n[5] Training machine learning models...")
print("-" * 80)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=50, max_depth=15, 
                                          random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, max_depth=5, 
                                                   learning_rate=0.1, random_state=42)
}

results = []

for model_name, model in models.items():
    # Train model
    print(f"\nTraining {model_name}...")
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Store results
    results.append({
        'Model': model_name,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train R²': train_r2,
        'Test R²': test_r2
    })
    
    print(f"  Train RMSE: ₹{train_rmse:.2f} | Test RMSE: ₹{test_rmse:.2f}")
    print(f"  Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")


[5] Training machine learning models...
--------------------------------------------------------------------------------

Training Linear Regression...
  Train RMSE: ₹7000.71 | Test RMSE: ₹7006.79
  Train R²: 0.9049 | Test R²: 0.9048

Training Ridge Regression...
  Train RMSE: ₹7000.71 | Test RMSE: ₹7006.79
  Train R²: 0.9049 | Test R²: 0.9048

Training Random Forest...
  Train RMSE: ₹2246.40 | Test RMSE: ₹2663.62
  Train R²: 0.9902 | Test R²: 0.9862

Training Gradient Boosting...
  Train RMSE: ₹4201.17 | Test RMSE: ₹4255.92
  Train R²: 0.9657 | Test R²: 0.9649


## 7. Model Comparison Results

In [7]:
print("\n" + "="*80)
print("MODEL COMPARISON RESULTS")
print("="*80)

results_df = pd.DataFrame(results)
print("\n", results_df.to_string(index=False))

# Find best model
best_model_idx = results_df['Test RMSE'].idxmin()
best_model_name = results_df.iloc[best_model_idx]['Model']
best_rmse = results_df.iloc[best_model_idx]['Test RMSE']
best_r2 = results_df.iloc[best_model_idx]['Test R²']

print("\n" + "-" * 80)
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Test RMSE: ₹{best_rmse:.2f}")
print(f"   Test R²: {best_r2:.4f}")
print("-" * 80)


MODEL COMPARISON RESULTS

             Model   Train MAE    Test MAE  Train RMSE   Test RMSE  Train R²  Test R²
Linear Regression 4640.839014 4618.812048 7000.706899 7006.786933  0.904856 0.904759
 Ridge Regression 4640.880448 4618.851264 7000.706912 7006.786082  0.904856 0.904759
    Random Forest 1126.689032 1293.674521 2246.397253 2663.617828  0.990204 0.986236
Gradient Boosting 2455.820777 2467.848875 4201.174774 4255.924597  0.965736 0.964862

--------------------------------------------------------------------------------
🏆 BEST MODEL: Random Forest
   Test RMSE: ₹2663.62
   Test R²: 0.9862
--------------------------------------------------------------------------------


## 8. Sample Predictions with Best Model

In [8]:
print("\n[6] Sample Predictions (Best Model)...")
print("-" * 80)

best_model = models[best_model_name]
sample_indices = np.random.choice(len(X_test), 5, replace=False)

predictions_summary = []

for idx in sample_indices:
    actual_price = y_test.iloc[idx]
    predicted_price = best_model.predict(X_test.iloc[[idx]])[0]
    error = abs(actual_price - predicted_price)
    error_pct = (error / actual_price) * 100
    
    print(f"\nSample {idx}:")
    print(f"  Actual Price:    ₹{actual_price:,.2f}")
    print(f"  Predicted Price: ₹{predicted_price:,.2f}")
    print(f"  Error:           ₹{error:,.2f} ({error_pct:.2f}%)")
    
    predictions_summary.append({
        'Sample Index': idx,
        'Actual Price': actual_price,
        'Predicted Price': predicted_price,
        'Absolute Error': error,
        'Error %': error_pct
    })

predictions_df = pd.DataFrame(predictions_summary)
print("\n" + "="*80)
predictions_df


[6] Sample Predictions (Best Model)...
--------------------------------------------------------------------------------

Sample 1711:
  Actual Price:    ₹5,960.00
  Predicted Price: ₹6,485.58
  Error:           ₹525.58 (8.82%)

Sample 34939:
  Actual Price:    ₹5,915.00
  Predicted Price: ₹5,512.22
  Error:           ₹402.78 (6.81%)

Sample 6089:
  Actual Price:    ₹10,680.00
  Predicted Price: ₹10,958.38
  Error:           ₹278.38 (2.61%)

Sample 57132:
  Actual Price:    ₹3,654.00
  Predicted Price: ₹4,602.36
  Error:           ₹948.36 (25.95%)

Sample 36778:
  Actual Price:    ₹7,783.00
  Predicted Price: ₹5,735.48
  Error:           ₹2,047.52 (26.31%)



,Sample Index,Actual Price,Predicted Price,Absolute Error,Error %
0,1711,5960,6485.584765,525.584765,8.818536
1,34939,5915,5512.223684,402.776316,6.809405
2,6089,10680,10958.384407,278.384407,2.606596
3,57132,3654,4602.359641,948.359641,25.954013
4,36778,7783,5735.475153,2047.524847,26.307656


## 9. Save Results

In [9]:
print("\n[7] Saving results...")
results_df.to_csv('model_training_results/sample_training_results.csv', index=False)
print("✓ Results saved to model_training_results/sample_training_results.csv")

print("\n" + "="*80)
print("✓ TRAINING COMPLETE!")
print("="*80)


[7] Saving results...
✓ Results saved to model_training_results/sample_training_results.csv

✓ TRAINING COMPLETE!


## Summary

### Key Findings:
- **Best Model**: Random Forest with Test R² = 0.9862
- **Best Test RMSE**: ₹2,663.62
- **Dataset**: 240,122 training samples, 60,031 test samples
- **Features**: 10 categorical, 2 numerical features
- **Target Range**: ₹1,105 to ₹123,071

The Random Forest model demonstrates excellent predictive performance with ~98.62% variance explained in the test data.